In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('bi.csv')

print("Dataset Shape")
print(df.shape)

print("\nFirst 5 Rows")
display(df.head())

print("\n--- Missing Values Check ---")
print(df.isnull().sum())

Dataset Shape
(77, 11)

First 5 Rows


,fNAME,lNAME,Age,gender,country,residence,entryEXAM,prevEducation,studyHOURS,Python,DB
0,Christina,Binger,44,Female,Norway,Private,72,Masters,158,59.0,55
1,Alex,Walekhwa,60,M,Kenya,Private,79,Diploma,150,60.0,75
2,Philip,Leo,25,Male,Uganda,Sognsvann,55,HighSchool,130,74.0,50
3,Shoni,Hlongwane,22,F,Rsa,Sognsvann,40,High School,120,NaN,44
4,Maria,Kedibone,23,Female,South Africa,Sognsvann,65,High School,122,91.0,80



--- Missing Values Check ---
fNAME            0
lNAME            0
Age              0
gender           0
country          0
residence        0
entryEXAM        0
prevEducation    0
studyHOURS       0
Python           2
DB               0
dtype: int64


In [ ]:
# Make a copy for processing
df_clean = df.copy()

# Fill missing numerical values with median, categorical with mode
for col in df_clean.select_dtypes(include=[np.number]).columns:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)

for col in df_clean.select_dtypes(include=['object']).columns:
    df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

# Feature Engineering: Create an overall average performance score if separate Python and DB scores exist
# (Update column names below if your dataset uses different exact names)
score_cols = [col for col in df_clean.columns if 'score' in col.lower() or 'grade' in col.lower() or 'python' in col.lower() or 'db' in col.lower()]
print("Identified Score Columns:", score_cols)

# Example: If you have Python and Database columns, compute overall average performance
# df_clean['Overall_Performance'] = df_clean[['Python_Score', 'DB_Score']].mean(axis=1)

Identified Score Columns: ['Python', 'DB']


/tmp/ipykernel_7765/2937931687.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean[col].fillna(df_clean[col].median(), inplace=True)
/tmp/ipykernel_7765/2937931687.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=Tr

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

# Prepare data for modeling
model_df = df_clean.copy()

# Encode categorical columns (like gender, country, residence, prevEducation) into numerical values
le = LabelEncoder()
for col in model_df.select_dtypes(include=['object']).columns:
    model_df[col] = le.fit_transform(model_df[col])

# Define Target (e.g., 'Python' or 'DB') and Features
target_col = 'Python' # Change to 'DB' if you want to analyze database scores instead

# Exclude names and target score columns from features
drop_cols = ['fNAME', 'lNAME', target_col]
# Also drop 'DB' from features if we are predicting 'Python' to prevent data leakage
if 'DB' in model_df.columns and target_col == 'Python':
    drop_cols.append('DB')
elif 'Python' in model_df.columns and target_col == 'DB':
    drop_cols.append('Python')

features = [col for col in model_df.columns if col not in drop_cols]

X = model_df[features]
y = model_df[target_col]

# Train Random Forest to check what actually predicts student success
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X, y)

# Extract Feature Importances
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("--- Feature Importances for Admissions Analysis ---")
display(importance_df)

--- Feature Importances for Admissions Analysis ---


,Feature,Importance
6,studyHOURS,0.668013
4,entryEXAM,0.188608
0,Age,0.059427
3,residence,0.028451
2,country,0.025819
1,gender,0.022028
5,prevEducation,0.007654


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Select numerical attributes for student profiling
clustering_features = ['studyHOURS', 'entryEXAM', 'Age']
clustering_features = [col for col in clustering_features if col in df_clean.columns]

if len(clustering_features) >= 2:
    X_cluster = df_clean[clustering_features]

    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_cluster)

    # Apply K-Means (Segmenting students into 3 profiles)
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    df_clean['Student_Segment'] = kmeans.fit_predict(X_scaled)

    # Map clusters to readable labels
    segment_names = {0: 'Moderate Performers', 1: 'High-Risk / At-Risk', 2: 'High Performers'}
    df_clean['Student_Segment'] = df_clean['Student_Segment'].map(segment_names)

    print("--- Student Segmentation Complete ---")
    print(df_clean['Student_Segment'].value_counts())
else:
    print("Check your column names for clustering features.")

--- Student Segmentation Complete ---
Student_Segment
Moderate Performers    32
High Performers        27
High-Risk / At-Risk    18
Name: count, dtype: int64


In [ ]:
# Save the enriched dataset as a CSV file ready for Power BI
output_filename = 'walsoft_enriched_student_data.csv'
df_clean.to_csv(output_filename, index=False)

print(f"File successfully saved as '{output_filename}'. Download this file from your Colab file explorer (left sidebar folder icon) and import it straight into Power BI!")

File successfully saved as 'walsoft_enriched_student_data.csv'. Download this file from your Colab file explorer (left sidebar folder icon) and import it straight into Power BI!
